In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q qdrant-client weaviate-client
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.3 MB/s eta 0:00:00


In [2]:
from bs4 import BeautifulSoup
import re

# ====================  TEXT EXTRACTION ====================
def extract_text_from_htm(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        raw_text = soup.get_text(separator='\n')

        lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
        text = '\n'.join(lines)
        clean_text = re.sub(r'\n\s*\n+', '\n\n', text).strip()
        return clean_text
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Cunk_size = 256
# Overlap = 32

In [6]:
import os
import time
import gc
import shutil
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== CONFIG =====================
books_folder   = "/content/drive/MyDrive/100_books/"
model_name     = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qdrant_path    = "/content/qdrant_storage"
weaviate_path  = "/content/weaviate_storage"

# SINGLE CONFIG (no loop)
chunk_size     = 512
chunk_overlap  = 64

# ===================== HELPER: extract text from HTM =====================
def extract_text_from_htm(file_path):
    """Extract text from HTML/HTM file."""
    from bs4 import BeautifulSoup
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
        for script in soup(["script", "style"]):
            script.decompose()
        return soup.get_text(separator="\n", strip=True)

# ===================== HELPER: get folder size =====================
def get_folder_size(path):
    """Returns total size of a folder in bytes"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== LOAD BOOKS =====================
htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
print(f"Found {len(htm_files)} books\n")

# ===================== SINGLE EXPERIMENT =====================
print(f"\n{'='*70}")
print(f"TESTING: chunk_size={chunk_size}, overlap={chunk_overlap}")
print(f"{'='*70}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

# ===================== CHUNK + ENCODE =====================
all_chunks     = []
all_embeddings = []
total_encode_time = 0.0

for file_name in tqdm(htm_files, desc="Chunking & Embedding"):
    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    if len(text.strip()) < 100:
        continue

    chunks = splitter.split_text(text)
    if not chunks:
        continue

    start = time.time()
    embeddings = model.encode(
        chunks,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    total_encode_time += time.time() - start

    all_chunks.extend(chunks)
    all_embeddings.append(embeddings)

    del embeddings
    torch.cuda.empty_cache()
    gc.collect()

all_embeddings_np = np.vstack(all_embeddings).astype('float32')
total_chunks = len(all_chunks)
emb_dim      = all_embeddings_np.shape[1]

# Pure embedding matrix size (no vector store overhead)
pure_emb_bytes = all_embeddings_np.nbytes
text_bytes     = sum(len(c.encode('utf-8')) for c in all_chunks)

print(f"  Created {total_chunks} chunks | Dim: {emb_dim}")

# ===================== QDRANT =====================
if os.path.exists(qdrant_path):
    shutil.rmtree(qdrant_path)

qdrant_client = QdrantClient(path=qdrant_path)
qdrant_collection = "test_collection"

qdrant_client.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=emb_dim, distance=Distance.COSINE)
)

qdrant_start = time.time()
batch_size = 512
for i in range(0, len(all_embeddings_np), batch_size):
    batch_end = min(i + batch_size, len(all_embeddings_np))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings_np[j].tolist(),
            payload={"text": all_chunks[j][:500]}
        )
        for j in range(i, batch_end)
    ]
    qdrant_client.upsert(collection_name=qdrant_collection, points=points)
qdrant_time = time.time() - qdrant_start

qdrant_client.close()
time.sleep(1)
qdrant_disk_bytes = get_folder_size(qdrant_path)

# ===================== WEAVIATE =====================
if os.path.exists(weaviate_path):
    shutil.rmtree(weaviate_path)

weaviate_client = weaviate.connect_to_embedded(
    persistence_data_path=weaviate_path
)
weaviate_collection = "TestCollection"

weaviate_client.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[Property(name="text", data_type=DataType.TEXT)]
)

col = weaviate_client.collections.get(weaviate_collection)

weaviate_start = time.time()
with col.batch.fixed_size(batch_size=512) as batch:
    for i, (vector, chunk) in enumerate(zip(all_embeddings_np, all_chunks)):
        batch.add_object(
            properties={"text": chunk[:500]},
            vector=vector.tolist()
        )
weaviate_time = time.time() - weaviate_start

weaviate_count = col.aggregate.over_all(total_count=True).total_count

weaviate_client.close()
time.sleep(1)
weaviate_disk_bytes = get_folder_size(weaviate_path)

# ===================== SAVE RESULT =====================
result = {
    "chunk_size":             chunk_size,
    "chunk_overlap":          chunk_overlap,
    "total_chunks":           total_chunks,
    "avg_chunks_per_book":    round(total_chunks / len(htm_files), 1),

    "encode_time_min":        round(total_encode_time / 60, 2),
    "chunks_per_second":      round(total_chunks / total_encode_time, 1),

    "pure_embedding_MB":      round(pure_emb_bytes / 1024**2, 2),
    "text_MB":                round(text_bytes / 1024**2, 2),

    "qdrant_index_time_sec":  round(qdrant_time, 2),
    "qdrant_disk_MB":         round(qdrant_disk_bytes / 1024**2, 2),
    "qdrant_overhead_MB":     round((qdrant_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_index_time_sec": round(weaviate_time, 2),
    "weaviate_disk_MB":        round(weaviate_disk_bytes / 1024**2, 2),
    "weaviate_overhead_MB":    round((weaviate_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_objects":        weaviate_count,
}

print(f"""
  Chunks:           {total_chunks}
  Pure embeddings:  {result['pure_embedding_MB']} MB
  Qdrant disk:      {result['qdrant_disk_MB']} MB (overhead: {result['qdrant_overhead_MB']} MB)
  Weaviate disk:    {result['weaviate_disk_MB']} MB (overhead: {result['weaviate_overhead_MB']} MB)
  Qdrant time:      {result['qdrant_index_time_sec']}s
  Weaviate time:    {result['weaviate_index_time_sec']}s
""")

# ===================== SAVE TO CSV =====================
df = pd.DataFrame([result])
df.to_csv("/content/drive/MyDrive/vdb_disk_comparison_single_config.csv", index=False)
print("Saved to vdb_disk_comparison_single_config.csv")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Found 100 books


TESTING: chunk_size=512, overlap=64


Chunking & Embedding:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Created 105123 chunks | Dim: 768


INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 21461



  Chunks:           105123
  Pure embeddings:  307.98 MB
  Qdrant disk:      825.74 MB (overhead: 517.76 MB)
  Weaviate disk:    588.42 MB (overhead: 280.44 MB)
  Qdrant time:      844.79s
  Weaviate time:    209.62s

Saved to vdb_disk_comparison_single_config.csv


# Cunk_size = 512
# Overlap = 64

In [7]:
import os
import time
import gc
import shutil
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== CONFIG =====================
books_folder   = "/content/drive/MyDrive/100_books/"
model_name     = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qdrant_path    = "/content/qdrant_storage"
weaviate_path  = "/content/weaviate_storage"

# SINGLE CONFIG (no loop)
chunk_size     = 256
chunk_overlap  = 32

# ===================== HELPER: extract text from HTM =====================
def extract_text_from_htm(file_path):
    """Extract text from HTML/HTM file."""
    from bs4 import BeautifulSoup
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
        for script in soup(["script", "style"]):
            script.decompose()
        return soup.get_text(separator="\n", strip=True)

# ===================== HELPER: get folder size =====================
def get_folder_size(path):
    """Returns total size of a folder in bytes"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== LOAD BOOKS =====================
htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
print(f"Found {len(htm_files)} books\n")

# ===================== SINGLE EXPERIMENT =====================
print(f"\n{'='*70}")
print(f"TESTING: chunk_size={chunk_size}, overlap={chunk_overlap}")
print(f"{'='*70}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

# ===================== CHUNK + ENCODE =====================
all_chunks     = []
all_embeddings = []
total_encode_time = 0.0

for file_name in tqdm(htm_files, desc="Chunking & Embedding"):
    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    if len(text.strip()) < 100:
        continue

    chunks = splitter.split_text(text)
    if not chunks:
        continue

    start = time.time()
    embeddings = model.encode(
        chunks,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    total_encode_time += time.time() - start

    all_chunks.extend(chunks)
    all_embeddings.append(embeddings)

    del embeddings
    torch.cuda.empty_cache()
    gc.collect()

all_embeddings_np = np.vstack(all_embeddings).astype('float32')
total_chunks = len(all_chunks)
emb_dim      = all_embeddings_np.shape[1]

# Pure embedding matrix size (no vector store overhead)
pure_emb_bytes = all_embeddings_np.nbytes
text_bytes     = sum(len(c.encode('utf-8')) for c in all_chunks)

print(f"  Created {total_chunks} chunks | Dim: {emb_dim}")

# ===================== QDRANT =====================
if os.path.exists(qdrant_path):
    shutil.rmtree(qdrant_path)

qdrant_client = QdrantClient(path=qdrant_path)
qdrant_collection = "test_collection"

qdrant_client.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=emb_dim, distance=Distance.COSINE)
)

qdrant_start = time.time()
batch_size = 512
for i in range(0, len(all_embeddings_np), batch_size):
    batch_end = min(i + batch_size, len(all_embeddings_np))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings_np[j].tolist(),
            payload={"text": all_chunks[j][:500]}
        )
        for j in range(i, batch_end)
    ]
    qdrant_client.upsert(collection_name=qdrant_collection, points=points)
qdrant_time = time.time() - qdrant_start

qdrant_client.close()
time.sleep(1)
qdrant_disk_bytes = get_folder_size(qdrant_path)

# ===================== WEAVIATE =====================
if os.path.exists(weaviate_path):
    shutil.rmtree(weaviate_path)

weaviate_client = weaviate.connect_to_embedded(
    persistence_data_path=weaviate_path
)
weaviate_collection = "TestCollection"

weaviate_client.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[Property(name="text", data_type=DataType.TEXT)]
)

col = weaviate_client.collections.get(weaviate_collection)

weaviate_start = time.time()
with col.batch.fixed_size(batch_size=512) as batch:
    for i, (vector, chunk) in enumerate(zip(all_embeddings_np, all_chunks)):
        batch.add_object(
            properties={"text": chunk[:500]},
            vector=vector.tolist()
        )
weaviate_time = time.time() - weaviate_start

weaviate_count = col.aggregate.over_all(total_count=True).total_count

weaviate_client.close()
time.sleep(1)
weaviate_disk_bytes = get_folder_size(weaviate_path)

# ===================== SAVE RESULT =====================
result = {
    "chunk_size":             chunk_size,
    "chunk_overlap":          chunk_overlap,
    "total_chunks":           total_chunks,
    "avg_chunks_per_book":    round(total_chunks / len(htm_files), 1),

    "encode_time_min":        round(total_encode_time / 60, 2),
    "chunks_per_second":      round(total_chunks / total_encode_time, 1),

    "pure_embedding_MB":      round(pure_emb_bytes / 1024**2, 2),
    "text_MB":                round(text_bytes / 1024**2, 2),

    "qdrant_index_time_sec":  round(qdrant_time, 2),
    "qdrant_disk_MB":         round(qdrant_disk_bytes / 1024**2, 2),
    "qdrant_overhead_MB":     round((qdrant_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_index_time_sec": round(weaviate_time, 2),
    "weaviate_disk_MB":        round(weaviate_disk_bytes / 1024**2, 2),
    "weaviate_overhead_MB":    round((weaviate_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_objects":        weaviate_count,
}

print(f"""
  Chunks:           {total_chunks}
  Pure embeddings:  {result['pure_embedding_MB']} MB
  Qdrant disk:      {result['qdrant_disk_MB']} MB (overhead: {result['qdrant_overhead_MB']} MB)
  Weaviate disk:    {result['weaviate_disk_MB']} MB (overhead: {result['weaviate_overhead_MB']} MB)
  Qdrant time:      {result['qdrant_index_time_sec']}s
  Weaviate time:    {result['weaviate_index_time_sec']}s
""")

# ===================== SAVE TO CSV =====================
df = pd.DataFrame([result])
df.to_csv("/content/drive/MyDrive/vdb_disk_comparison_single_config.csv", index=False)
print("Saved to vdb_disk_comparison_single_config.csv")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Found 100 books


TESTING: chunk_size=256, overlap=32


Chunking & Embedding:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Created 230141 chunks | Dim: 768


INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 34487



  Chunks:           230141
  Pure embeddings:  674.24 MB
  Qdrant disk:      1808.22 MB (overhead: 1133.98 MB)
  Weaviate disk:    998.93 MB (overhead: 324.69 MB)
  Qdrant time:      1935.82s
  Weaviate time:    557.0s

Saved to vdb_disk_comparison_single_config.csv


# Cunk_size = 600
# Overlap = 90

In [9]:
import os
import time
import gc
import shutil
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== CONFIG =====================
books_folder   = "/content/drive/MyDrive/100_books/"
model_name     = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qdrant_path    = "/content/qdrant_storage"
weaviate_path  = "/content/weaviate_storage"

# SINGLE CONFIG (no loop)
chunk_size     = 600
chunk_overlap  = 90

# ===================== HELPER: extract text from HTM =====================
def extract_text_from_htm(file_path):
    """Extract text from HTML/HTM file."""
    from bs4 import BeautifulSoup
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
        for script in soup(["script", "style"]):
            script.decompose()
        return soup.get_text(separator="\n", strip=True)

# ===================== HELPER: get folder size =====================
def get_folder_size(path):
    """Returns total size of a folder in bytes"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== LOAD BOOKS =====================
htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
print(f"Found {len(htm_files)} books\n")

# ===================== SINGLE EXPERIMENT =====================
print(f"\n{'='*70}")
print(f"TESTING: chunk_size={chunk_size}, overlap={chunk_overlap}")
print(f"{'='*70}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

# ===================== CHUNK + ENCODE =====================
all_chunks     = []
all_embeddings = []
total_encode_time = 0.0

for file_name in tqdm(htm_files, desc="Chunking & Embedding"):
    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    if len(text.strip()) < 100:
        continue

    chunks = splitter.split_text(text)
    if not chunks:
        continue

    start = time.time()
    embeddings = model.encode(
        chunks,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    total_encode_time += time.time() - start

    all_chunks.extend(chunks)
    all_embeddings.append(embeddings)

    del embeddings
    torch.cuda.empty_cache()
    gc.collect()

all_embeddings_np = np.vstack(all_embeddings).astype('float32')
total_chunks = len(all_chunks)
emb_dim      = all_embeddings_np.shape[1]

# Pure embedding matrix size (no vector store overhead)
pure_emb_bytes = all_embeddings_np.nbytes
text_bytes     = sum(len(c.encode('utf-8')) for c in all_chunks)

print(f"  Created {total_chunks} chunks | Dim: {emb_dim}")

# ===================== QDRANT =====================
if os.path.exists(qdrant_path):
    shutil.rmtree(qdrant_path)

qdrant_client = QdrantClient(path=qdrant_path)
qdrant_collection = "test_collection"

qdrant_client.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=emb_dim, distance=Distance.COSINE)
)

qdrant_start = time.time()
batch_size = 512
for i in range(0, len(all_embeddings_np), batch_size):
    batch_end = min(i + batch_size, len(all_embeddings_np))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings_np[j].tolist(),
            payload={"text": all_chunks[j][:500]}
        )
        for j in range(i, batch_end)
    ]
    qdrant_client.upsert(collection_name=qdrant_collection, points=points)
qdrant_time = time.time() - qdrant_start

qdrant_client.close()
time.sleep(1)
qdrant_disk_bytes = get_folder_size(qdrant_path)

# ===================== WEAVIATE =====================
if os.path.exists(weaviate_path):
    shutil.rmtree(weaviate_path)

weaviate_client = weaviate.connect_to_embedded(
    persistence_data_path=weaviate_path
)
weaviate_collection = "TestCollection"

weaviate_client.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[Property(name="text", data_type=DataType.TEXT)]
)

col = weaviate_client.collections.get(weaviate_collection)

weaviate_start = time.time()
with col.batch.fixed_size(batch_size=512) as batch:
    for i, (vector, chunk) in enumerate(zip(all_embeddings_np, all_chunks)):
        batch.add_object(
            properties={"text": chunk[:500]},
            vector=vector.tolist()
        )
weaviate_time = time.time() - weaviate_start

weaviate_count = col.aggregate.over_all(total_count=True).total_count

weaviate_client.close()
time.sleep(1)
weaviate_disk_bytes = get_folder_size(weaviate_path)

# ===================== SAVE RESULT =====================
result = {
    "chunk_size":             chunk_size,
    "chunk_overlap":          chunk_overlap,
    "total_chunks":           total_chunks,
    "avg_chunks_per_book":    round(total_chunks / len(htm_files), 1),

    "encode_time_min":        round(total_encode_time / 60, 2),
    "chunks_per_second":      round(total_chunks / total_encode_time, 1),

    "pure_embedding_MB":      round(pure_emb_bytes / 1024**2, 2),
    "text_MB":                round(text_bytes / 1024**2, 2),

    "qdrant_index_time_sec":  round(qdrant_time, 2),
    "qdrant_disk_MB":         round(qdrant_disk_bytes / 1024**2, 2),
    "qdrant_overhead_MB":     round((qdrant_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_index_time_sec": round(weaviate_time, 2),
    "weaviate_disk_MB":        round(weaviate_disk_bytes / 1024**2, 2),
    "weaviate_overhead_MB":    round((weaviate_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_objects":        weaviate_count,
}

print(f"""
  Chunks:           {total_chunks}
  Pure embeddings:  {result['pure_embedding_MB']} MB
  Qdrant disk:      {result['qdrant_disk_MB']} MB (overhead: {result['qdrant_overhead_MB']} MB)
  Weaviate disk:    {result['weaviate_disk_MB']} MB (overhead: {result['weaviate_overhead_MB']} MB)
  Qdrant time:      {result['qdrant_index_time_sec']}s
  Weaviate time:    {result['weaviate_index_time_sec']}s
""")

# ===================== SAVE TO CSV =====================
df = pd.DataFrame([result])
df.to_csv("/content/drive/MyDrive/vdb_disk_comparison_single_config.csv", index=False)
print("Saved to vdb_disk_comparison_single_config.csv")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Found 100 books


TESTING: chunk_size=600, overlap=90


Chunking & Embedding:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Created 88275 chunks | Dim: 768


INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 43787



  Chunks:           88275
  Pure embeddings:  258.62 MB
  Qdrant disk:      693.34 MB (overhead: 434.72 MB)
  Weaviate disk:    490.34 MB (overhead: 231.72 MB)
  Qdrant time:      712.53s
  Weaviate time:    179.11s

Saved to vdb_disk_comparison_single_config.csv


# Cunk_size = 1024
# Overlap = 128

In [ ]:
import os
import time
import gc
import shutil
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== CONFIG =====================
books_folder   = "/content/drive/MyDrive/100_books/"
model_name     = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qdrant_path    = "/content/qdrant_storage"
weaviate_path  = "/content/weaviate_storage"

# SINGLE CONFIG (no loop)
chunk_size     = 1024
chunk_overlap  = 128

# ===================== HELPER: extract text from HTM =====================
def extract_text_from_htm(file_path):
    """Extract text from HTML/HTM file."""
    from bs4 import BeautifulSoup
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
        for script in soup(["script", "style"]):
            script.decompose()
        return soup.get_text(separator="\n", strip=True)

# ===================== HELPER: get folder size =====================
def get_folder_size(path):
    """Returns total size of a folder in bytes"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== LOAD BOOKS =====================
htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
print(f"Found {len(htm_files)} books\n")

# ===================== SINGLE EXPERIMENT =====================
print(f"\n{'='*70}")
print(f"TESTING: chunk_size={chunk_size}, overlap={chunk_overlap}")
print(f"{'='*70}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

# ===================== CHUNK + ENCODE =====================
all_chunks     = []
all_embeddings = []
total_encode_time = 0.0

for file_name in tqdm(htm_files, desc="Chunking & Embedding"):
    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    if len(text.strip()) < 100:
        continue

    chunks = splitter.split_text(text)
    if not chunks:
        continue

    start = time.time()
    embeddings = model.encode(
        chunks,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    total_encode_time += time.time() - start

    all_chunks.extend(chunks)
    all_embeddings.append(embeddings)

    del embeddings
    torch.cuda.empty_cache()
    gc.collect()

all_embeddings_np = np.vstack(all_embeddings).astype('float32')
total_chunks = len(all_chunks)
emb_dim      = all_embeddings_np.shape[1]

# Pure embedding matrix size (no vector store overhead)
pure_emb_bytes = all_embeddings_np.nbytes
text_bytes     = sum(len(c.encode('utf-8')) for c in all_chunks)

print(f"  Created {total_chunks} chunks | Dim: {emb_dim}")

# ===================== QDRANT =====================
if os.path.exists(qdrant_path):
    shutil.rmtree(qdrant_path)

qdrant_client = QdrantClient(path=qdrant_path)
qdrant_collection = "test_collection"

qdrant_client.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=emb_dim, distance=Distance.COSINE)
)

qdrant_start = time.time()
batch_size = 512
for i in range(0, len(all_embeddings_np), batch_size):
    batch_end = min(i + batch_size, len(all_embeddings_np))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings_np[j].tolist(),
            payload={"text": all_chunks[j][:500]}
        )
        for j in range(i, batch_end)
    ]
    qdrant_client.upsert(collection_name=qdrant_collection, points=points)
qdrant_time = time.time() - qdrant_start

qdrant_client.close()
time.sleep(1)
qdrant_disk_bytes = get_folder_size(qdrant_path)

# ===================== WEAVIATE =====================
if os.path.exists(weaviate_path):
    shutil.rmtree(weaviate_path)

weaviate_client = weaviate.connect_to_embedded(
    persistence_data_path=weaviate_path
)
weaviate_collection = "TestCollection"

weaviate_client.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[Property(name="text", data_type=DataType.TEXT)]
)

col = weaviate_client.collections.get(weaviate_collection)

weaviate_start = time.time()
with col.batch.fixed_size(batch_size=512) as batch:
    for i, (vector, chunk) in enumerate(zip(all_embeddings_np, all_chunks)):
        batch.add_object(
            properties={"text": chunk[:500]},
            vector=vector.tolist()
        )
weaviate_time = time.time() - weaviate_start

weaviate_count = col.aggregate.over_all(total_count=True).total_count

weaviate_client.close()
time.sleep(1)
weaviate_disk_bytes = get_folder_size(weaviate_path)

# ===================== SAVE RESULT =====================
result = {
    "chunk_size":             chunk_size,
    "chunk_overlap":          chunk_overlap,
    "total_chunks":           total_chunks,
    "avg_chunks_per_book":    round(total_chunks / len(htm_files), 1),

    "encode_time_min":        round(total_encode_time / 60, 2),
    "chunks_per_second":      round(total_chunks / total_encode_time, 1),

    "pure_embedding_MB":      round(pure_emb_bytes / 1024**2, 2),
    "text_MB":                round(text_bytes / 1024**2, 2),

    "qdrant_index_time_sec":  round(qdrant_time, 2),
    "qdrant_disk_MB":         round(qdrant_disk_bytes / 1024**2, 2),
    "qdrant_overhead_MB":     round((qdrant_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_index_time_sec": round(weaviate_time, 2),
    "weaviate_disk_MB":        round(weaviate_disk_bytes / 1024**2, 2),
    "weaviate_overhead_MB":    round((weaviate_disk_bytes - pure_emb_bytes) / 1024**2, 2),

    "weaviate_objects":        weaviate_count,
}

print(f"""
  Chunks:           {total_chunks}
  Pure embeddings:  {result['pure_embedding_MB']} MB
  Qdrant disk:      {result['qdrant_disk_MB']} MB (overhead: {result['qdrant_overhead_MB']} MB)
  Weaviate disk:    {result['weaviate_disk_MB']} MB (overhead: {result['weaviate_overhead_MB']} MB)
  Qdrant time:      {result['qdrant_index_time_sec']}s
  Weaviate time:    {result['weaviate_index_time_sec']}s
""")

# ===================== SAVE TO CSV =====================
df = pd.DataFrame([result])
df.to_csv("/content/drive/MyDrive/vdb_disk_comparison_single_config.csv", index=False)
print("Saved to vdb_disk_comparison_single_config.csv")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Found 100 books


TESTING: chunk_size=1024, overlap=128


Chunking & Embedding:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [5]:
import os
import time
import gc
import shutil
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== CONFIG =====================
books_folder   = "/content/drive/MyDrive/100_books/"
model_name     = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qdrant_path    = "/content/qdrant_storage"    # persistent path for disk measurement
weaviate_path  = "/content/weaviate_storage"  # persistent path for disk measurement

chunk_configs = [
    {"chunk_size": 256,  "chunk_overlap": 32},
    {"chunk_size": 512,  "chunk_overlap": 64},
    {"chunk_size": 600,  "chunk_overlap": 90},
    {"chunk_size": 1024, "chunk_overlap": 128},
]

# ===================== HELPER: get folder size =====================
def get_folder_size(path):
    """Returns total size of a folder in bytes"""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== LOAD BOOKS =====================
htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
print(f"Found {len(htm_files)} books\n")

# ===================== MAIN EXPERIMENT =====================
final_results = []

for config in chunk_configs:
    chunk_size    = config["chunk_size"]
    chunk_overlap = config["chunk_overlap"]

    print(f"\n{'='*70}")
    print(f"TESTING: chunk_size={chunk_size}, overlap={chunk_overlap}")
    print(f"{'='*70}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
    )

    # ===================== CHUNK + ENCODE =====================
    all_chunks    = []
    all_embeddings = []
    total_encode_time = 0.0

    for file_name in tqdm(htm_files, desc="Chunking & Embedding"):
        file_path = os.path.join(books_folder, file_name)
        text = extract_text_from_htm(file_path)

        if len(text.strip()) < 100:
            continue

        chunks = splitter.split_text(text)
        if not chunks:
            continue

        start = time.time()
        embeddings = model.encode(
            chunks,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        total_encode_time += time.time() - start

        all_chunks.extend(chunks)
        all_embeddings.append(embeddings)

        del embeddings
        torch.cuda.empty_cache()
        gc.collect()

    all_embeddings_np = np.vstack(all_embeddings).astype('float32')
    total_chunks = len(all_chunks)
    emb_dim      = all_embeddings_np.shape[1]

    # Pure embedding matrix size (no vector store overhead)
    pure_emb_bytes = all_embeddings_np.nbytes
    text_bytes     = sum(len(c.encode('utf-8')) for c in all_chunks)

    print(f"  Created {total_chunks} chunks | Dim: {emb_dim}")

    # ===================== QDRANT (persistent path) =====================
    # Clean previous storage
    if os.path.exists(qdrant_path):
        shutil.rmtree(qdrant_path)

    qdrant_client = QdrantClient(path=qdrant_path)   # ← persistent, measurable
    qdrant_collection = "test_collection"

    qdrant_client.create_collection(
        collection_name=qdrant_collection,
        vectors_config=VectorParams(size=emb_dim, distance=Distance.COSINE)
    )

    qdrant_start = time.time()
    batch_size = 512
    for i in range(0, len(all_embeddings_np), batch_size):
        batch_end = min(i + batch_size, len(all_embeddings_np))
        points = [
            PointStruct(
                id=j,
                vector=all_embeddings_np[j].tolist(),
                payload={"text": all_chunks[j][:500]}
            )
            for j in range(i, batch_end)
        ]
        qdrant_client.upsert(collection_name=qdrant_collection, points=points)
    qdrant_time = time.time() - qdrant_start

    # Force flush before measuring
    qdrant_client.close()
    time.sleep(1)
    qdrant_disk_bytes = get_folder_size(qdrant_path)

    # ===================== WEAVIATE (embedded mode) =====================
    # Clean previous storage
    if os.path.exists(weaviate_path):
        shutil.rmtree(weaviate_path)

    weaviate_client = weaviate.connect_to_embedded(
        persistence_data_path=weaviate_path   # ← persistent, measurable
    )
    weaviate_collection = "TestCollection"

    weaviate_client.collections.create(
        name=weaviate_collection,
        vectorizer_config=Configure.Vectorizer.none(),
        properties=[Property(name="text", data_type=DataType.TEXT)]
    )

    col = weaviate_client.collections.get(weaviate_collection)

    weaviate_start = time.time()
    with col.batch.fixed_size(batch_size=512) as batch:
        for i, (vector, chunk) in enumerate(zip(all_embeddings_np, all_chunks)):
            batch.add_object(
                properties={"text": chunk[:500]},
                vector=vector.tolist()
            )
    weaviate_time = time.time() - weaviate_start

    weaviate_count = col.aggregate.over_all(total_count=True).total_count

    # Force flush before measuring
    weaviate_client.close()
    time.sleep(1)
    weaviate_disk_bytes = get_folder_size(weaviate_path)

    # ===================== SAVE RESULT =====================
    result = {
        "chunk_size":             chunk_size,
        "chunk_overlap":          chunk_overlap,
        "total_chunks":           total_chunks,
        "avg_chunks_per_book":    round(total_chunks / len(htm_files), 1),

        # Encoding
        "encode_time_min":        round(total_encode_time / 60, 2),
        "chunks_per_second":      round(total_chunks / total_encode_time, 1),

        # Pure embedding (no vector store overhead)
        "pure_embedding_MB":      round(pure_emb_bytes / 1024**2, 2),
        "text_MB":                round(text_bytes / 1024**2, 2),

        # Qdrant
        "qdrant_index_time_sec":  round(qdrant_time, 2),
        "qdrant_disk_MB":         round(qdrant_disk_bytes / 1024**2, 2),
        "qdrant_overhead_MB":     round((qdrant_disk_bytes - pure_emb_bytes) / 1024**2, 2),

        # Weaviate
        "weaviate_index_time_sec": round(weaviate_time, 2),
        "weaviate_disk_MB":        round(weaviate_disk_bytes / 1024**2, 2),
        "weaviate_overhead_MB":    round((weaviate_disk_bytes - pure_emb_bytes) / 1024**2, 2),

        "weaviate_objects":        weaviate_count,
    }

    final_results.append(result)

    print(f"""
  Chunks:           {total_chunks}
  Pure embeddings:  {result['pure_embedding_MB']} MB
  Qdrant disk:      {result['qdrant_disk_MB']} MB (overhead: {result['qdrant_overhead_MB']} MB)
  Weaviate disk:    {result['weaviate_disk_MB']} MB (overhead: {result['weaviate_overhead_MB']} MB)
  Qdrant time:      {result['qdrant_index_time_sec']}s
  Weaviate time:    {result['weaviate_index_time_sec']}s
""")

    # Clean up for next iteration
    del all_chunks, all_embeddings, all_embeddings_np
    torch.cuda.empty_cache()
    gc.collect()

# ===================== FINAL REPORT =====================
df = pd.DataFrame(final_results)
print("\n" + "="*100)
print("FINAL COMPARISON: CHUNK SIZE + QDRANT vs WEAVIATE DISK USAGE")
print("="*100)
print(df[[
    "chunk_size", "total_chunks",
    "pure_embedding_MB",
    "qdrant_disk_MB", "qdrant_overhead_MB", "qdrant_index_time_sec",
    "weaviate_disk_MB", "weaviate_overhead_MB", "weaviate_index_time_sec",
    "encode_time_min"
]].to_string(index=False))

df.to_csv("/content/drive/MyDrive/vdb_disk_comparison_chunk_sizes.csv", index=False)
print("\nSaved to vdb_disk_comparison_chunk_sizes.csv")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Found 100 books


TESTING: chunk_size=256, overlap=32


Chunking & Embedding:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  Created 230141 chunks | Dim: 768


/tmp/ipykernel_4732/1718892603.py:137: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client.upsert(collection_name=qdrant_collection, points=points)


KeyboardInterrupt: 